# 03 - Raw Four-Table Join and Data Quality




## 1. Setup



In [1]:
%pip install psycopg

^C
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from rapidfuzz import fuzz, process

In [1]:
from pathlib import Path
from getpass import getpass

import pandas as pd
import psycopg
from psycopg import sql
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Sesuaikan bila Anda membuat PostgreSQL role selain "postgres".
POSTGRES_HOST = "localhost"
POSTGRES_PORT = 5432
POSTGRES_USER = "postgres"

# Nama fisik database PostgreSQL: gunakan lowercase.
AML_DATABASE = "aml"

# Masukkan password ketika notebook meminta.
POSTGRES_PASSWORD = getpass("PostgreSQL password: ")

# Lokasi project dan data raw.
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"

assert RAW_DIR.exists(), f"Folder raw tidak ditemukan: {RAW_DIR}"

print(f"Project root : {ROOT}")
print(f"Raw CSV path : {RAW_DIR}")
print(f"Target DB    : {AML_DATABASE}")

Project root : E:\Trading\V-Teki Project\Anti Money Laundering Detection Updated
Raw CSV path : E:\Trading\V-Teki Project\Anti Money Laundering Detection Updated\data\raw
Target DB    : aml


## 2. Create database connection and load raw tables

Buat koneksi database ke postgresql

In [2]:
# CREATE DATABASE tidak boleh dijalankan di dalam transaction biasa,
# sehingga autocommit wajib aktif.
with psycopg.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    dbname="postgres",
    autocommit=True,
) as admin_connection:
    
    with admin_connection.cursor() as cursor:
        cursor.execute(
            "SELECT 1 FROM pg_database WHERE datname = %s",
            (AML_DATABASE,),
        )
        database_exists = cursor.fetchone() is not None

        if database_exists:
            print(f"Database '{AML_DATABASE}' sudah ada. Tidak dibuat ulang.")
        else:
            cursor.execute(
                sql.SQL("CREATE DATABASE {} ENCODING 'UTF8'").format(
                    sql.Identifier(AML_DATABASE)
                )
            )
            print(f"Database '{AML_DATABASE}' berhasil dibuat.")


Database 'aml' sudah ada. Tidak dibuat ulang.


In [3]:
aml_url = URL.create(
    drivername="postgresql+psycopg",
    username=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=AML_DATABASE,
)

engine = create_engine(
    aml_url,
    pool_pre_ping=True,
)

with engine.connect() as connection:
    database_check = connection.execute(
        text("""
            SELECT
                current_database() AS database_name,
                current_user AS database_user,
                version() AS postgresql_version
        """)
    ).mappings().one()

database_check

{'database_name': 'aml', 'database_user': 'postgres', 'postgresql_version': 'PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit'}

In [4]:
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))

print("Schema raw siap.")

Schema raw siap.


In [5]:
RAW_TABLES = {
    "accounts": {
        "file_name": "accounts.csv",
        "parse_dates": ["opening_date"],
    },
    "counterparties": {
        "file_name": "counterparties.csv",
        "parse_dates": [],
    },
    "customers": {
        "file_name": "customers.csv",
        "parse_dates": ["date_of_birth", "onboarding_date"],
    },
    "sanctions_watchlist": {
        "file_name": "sanctions_watchlist.csv",
        "parse_dates": ["listing_date"],
    },
    "transactions": {
        "file_name": "transactions.csv",
        "parse_dates": ["transaction_timestamp"],
    },
}

In [9]:
MAX_BIND_PARAMETERS = 60_000
MAX_ROWS_PER_INSERT = 1_000

load_summary = []

for table_name, spec in RAW_TABLES.items():
    csv_path = RAW_DIR / spec["file_name"]

    dataframe = pd.read_csv(
        csv_path,
        parse_dates=spec["parse_dates"],
    )

    # method='multi' sends rows x columns parameters per INSERT.
    # Keep each batch safely below PostgreSQL's 65,535-parameter limit.
    safe_chunksize = max(
        1,
        min(
            MAX_ROWS_PER_INSERT,
            MAX_BIND_PARAMETERS // len(dataframe.columns),
        ),
    )

    dataframe.to_sql(
        name=table_name,
        con=engine,
        schema="raw",
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=safe_chunksize,
    )

    load_summary.append(
        {
            "table": f"raw.{table_name}",
            "csv_rows": len(dataframe),
            "csv_columns": len(dataframe.columns),
            "insert_batch_rows": safe_chunksize,
        }
    )

load_summary_df = pd.DataFrame(load_summary)
display(load_summary_df)

,table,csv_rows,csv_columns,insert_batch_rows
0,raw.accounts,15000,11,1000
1,raw.counterparties,5000,16,1000
2,raw.customers,10000,26,1000
3,raw.sanctions_watchlist,1200,23,1000
4,raw.transactions,250000,32,1000


## 5. Join SQL Tables Menjadi Base Table




In [6]:
PROJECT_ROOT = Path(
    r"E:\Trading\V-Teki Project\Anti Money Laundering Detection Updated"
)

sql_path2 = PROJECT_ROOT / "sql" / "join_four_tables.sql"
sql_path_2 = sql_path2.read_text(encoding="utf-8").strip().rstrip(";")

joined_tables = pd.read_sql_query(
    sql=text(sql_path_2),
    con=engine,
)
display(joined_tables.head(8))



,transaction_id,transaction_timestamp,transaction_type,channel,transaction_status,debit_credit,amount,currency,amount_idr_equivalent,purpose_code,...,receiver_party_id,receiver_party_name,receiver_party_address,receiver_party_country,receiver_party_risk_level,receiver_customer_id,receiver_account_id,counterparty_id,beneficiary_name,beneficiary_address
0,TXN0000000001,2025-11-03 00:00:18,SWIFT,Internet,Success,Debit,6211901.22,IDR,6211901.22,OTHER,...,CUS0007767,Rizky Chandra,Jl. Prakoso No. 101,ID,Low,CUS0007767,ACC00003390,INTERNAL_ON_US_TRANSFER,Rizky Chandra,Jl. Prakoso No. 101
1,TXN0000000002,2025-11-03 00:02:26,RTGS,Branch,Success,Debit,1347876.32,IDR,1347876.32,INVESTMENT,...,CP0004049,Gita Adinata,101 Synthetic Avenue,ID,Medium,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0004049,Gita Adinata,101 Synthetic Avenue
2,TXN0000000003,2025-11-03 00:03:18,BI-FAST,ATM,Success,Debit,987476.37,IDR,987476.37,BILL,...,CP0003095,Raka Santoso,92 Synthetic Avenue,ID,Low,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0003095,Raka Santoso,92 Synthetic Avenue
3,TXN0000000004,2025-11-03 00:04:15,Cash,Mobile,Success,Debit,149.95,USD,2429199.44,FAMILY,...,CUS0000366,Indra Adinata,Jl. Gunawan No. 92,SG,Low,CUS0000366,ACC00008341,INTERNAL_ON_US_TRANSFER,Indra Adinata,Jl. Gunawan No. 92
4,TXN0000000005,2025-11-03 00:06:48,Transfer,API,Success,Debit,927217.72,IDR,927217.72,TRADE,...,CP0001839,Citra Santoso,593 Synthetic Avenue,ID,Low,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0001839,Citra Santoso,593 Synthetic Avenue
5,TXN0000000006,2025-11-03 00:09:02,Transfer,Branch,Success,Debit,2097778.34,IDR,2097778.34,INVESTMENT,...,CP0001201,Bima Kurniawan,385 Synthetic Avenue,ID,Medium,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0001201,Bima Kurniawan,385 Synthetic Avenue
6,TXN0000000007,2025-11-03 00:11:11,Cash,Mobile,Success,Debit,569967.57,IDR,569967.57,SALARY,...,CUS0003329,Farah Wibowo,Jl. Rahardjo No. 113,ID,Low,CUS0003329,ACC00007330,INTERNAL_ON_US_TRANSFER,Farah Wibowo,Jl. Rahardjo No. 113
7,TXN0000000008,2025-11-03 00:14:25,BI-FAST,ATM,Success,Debit,4899100.46,IDR,4899100.46,SALARY,...,CUS0005801,Anisa Iskandar,Jl. Rahardjo No. 168,ID,Low,CUS0005801,ACC00012038,INTERNAL_ON_US_TRANSFER,Anisa Iskandar,Jl. Rahardjo No. 168


# Proses Join yang saya lakukan:


Sesuai yang tadi di instruksikan fokus saya ke 4 tabel yaitu accounts, counterparties, customers, dan transactions. Karena fokus dari project AML itu adalah kita monitor transaksi maka transactions akan menjadi tabel utama maka dari itu grain/unit analisis saya adalah </br>
<b> Satu baris = satu transaction_id</b>
<br>
Karena tabel transactions berisi 250.000 transaksi. Untuk query dimulai dari <br>
<b> from raw.transactions as t</b> dengan tujuan hasil akhir tetap 250000 baris satu untuk setiap transaksi. Berikut tabel dan perannya:<br>
1. transactions = Kejadian transaksi: waktu, nominal, negara, device, pengirim, penerima dan sebagainya
2. customer = Profil/KYC customer bank: income, pekerjaan, PEP, risk rating.
3. accoutns = Profil rekening pengirim: tipe rekening, saldo, tanggal buka, risk level.
4. counterparties = Profil pihak eksternal yang bukan customer bank

Kalau ibu liat di file sql disitu saya ada melakukan join tabel customers dua kali. Tapi tabel tersebut punya dua peran berbeda. Misalkan:</br>
<b>
left join raw.customers as sender_customer
    on t.sender_customer_id = sender_customer.customer_id</b><br>
Untuk mengambil profil pengirim/sender
Dan satu lagi </br>
<b> left join raw.customers as receiver_customer
    on t.receiver_customer_id = receiver_customer.customer_id</b>
</br>
Itu untuk ambil profil penerima interna. Contoh:
</br>
Pengirim  : CUS0001001</br>
Penerima : CUS0002002
</br>
Terus untuk join dengan accounts saya pakai dua kondisi:
</br>
<b> left join raw.accounts as sender_account
    on t.sender_account_id = sender_account.account_id
    and t.sender_customer_id = sender_account.customer_id</b>
</br>
Tujuannya buat mastiin rekening tersebut benar-benar milik pengirim transaksi.
Tanpa kondisi kedua, secara teori kita bisa mengambil detail rekening yang ID-nya cocok tetapi pemiliknya tidak sesuai. Dengan dua kondisi ini, join menjadi lebih aman.
</br>
Terus bu tadi pagi kan waktu saya kasih liat hasil join kebetulan ada NaN. Itu ternyata setelah saya telusuri lebih dalam itu ada 2 faktor kondisi bisnis/transaksi berbeda: </br>

| Jenis transaksi | Profil customer penerima | Profil counterparty |
|---|---|---|
| Internal | Ada | Tidak ada |
| Eksternal | Tidak ada | Ada |

</br>
Contoh terjadi transfer internal:
receiver_customer_full_name = "Anisa Santoso"
counterparty_name = NaN
</br>

NaN itu terjadi gara2 misal contoh terjadi transfer internal ya automatis counterparty_name dan kolom2 bersangkutan akan jadi NaN di SQL. Begitu juga sebaliknya yang terjadi jika yang terjadi transfer external. 

</br>

Akhirnya saya pakai COALESCE karena untuk Null handling dan untuk punya kolom penerima yang lebih seragam contoh: 
- receiver_party_id
- receiver_party_name
- receiver_party_address
- receiver_party_country
- receiver_party_risk_level

COALESCE mengambil nilai pertama yang tidak kosong, dari kiri ke kanan.
Contoh:
coalesce(
    receiver_customer.full_name,
    counterparty.counterparty_name,
    t.receiver_name
) as receiver_party_name
Logikanya:
1. Jika penerima adalah customer internal, gunakan nama dari customers.
2. Jika bukan internal tetapi external counterparty tersedia, gunakan counterparties.counterparty_name.
3. Jika master data tidak tersedia, gunakan nama yang tercatat pada event transaksi: t.receiver_name.

Contoh nyata <b>COALESCE</b></br>
<b>Transfer internal</b></br>
receiver_customer.full_name     = "Anisa Santoso"
counterparty.counterparty_name  = NULL
t.receiver_name                 = "Anisa Santoso"
</br>
Hasil:
receiver_party_name = "Anisa Santoso"
</br>
<b>Transfer eksternal</b>
</br>
receiver_customer.full_name     = NULL
counterparty.counterparty_name  = "Rani Prakoso"
t.receiver_name                 = "Rani Prakoso"
Hasil: </br>
receiver_party_name = "Rani Prakoso"
Jadi kedua tipe transaksi sekarang punya satu kolom penerima yang konsisten.
</br>


In [13]:
joined_tables.columns

Index(['transaction_id', 'transaction_timestamp', 'transaction_type',
       'channel', 'transaction_status', 'debit_credit', 'amount', 'currency',
       'amount_idr_equivalent', 'purpose_code', 'purpose_description',
       'reference_number', 'source_of_fund', 'destination_bank',
       'destination_country', 'device_id', 'ip_address', 'latitude',
       'longitude', 'sender_customer_id', 'sender_account_id',
       'sender_customer_name', 'sender_customer_address',
       'sender_customer_country', 'sender_customer_monthly_income',
       'sender_customer_occupation', 'sender_customer_segment',
       'sender_customer_risk_rating', 'sender_customer_pep_flag',
       'sender_customer_onboarding_date', 'sender_account_type',
       'sender_account_currency', 'sender_branch_code',
       'sender_account_opening_date', 'sender_account_status',
       'sender_account_risk_level', 'receiver_party_id', 'receiver_party_name',
       'receiver_party_address', 'receiver_party_country',
     

In [11]:
view = ["sender_customer_monthly_income", "sender_customer_id", "sender_customer_risk_rating", "sender_customer_pep_flag",  "receiver_customer_id", "receiver_account_id", "receiver_party_name", "receiver_party_id"]
joined_tables[view].head(10)

,sender_customer_monthly_income,sender_customer_id,sender_customer_risk_rating,sender_customer_pep_flag,receiver_customer_id,receiver_account_id,receiver_party_name,receiver_party_id
0,5001000.0,CUS0004481,Low,0,CUS0007767,ACC00003390,Rizky Chandra,CUS0007767
1,6368000.0,CUS0007453,High,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Gita Adinata,CP0004049
2,18653000.0,CUS0005894,Medium,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Raka Santoso,CP0003095
3,10614000.0,CUS0000187,Low,0,CUS0000366,ACC00008341,Indra Adinata,CUS0000366
4,14113000.0,CUS0004923,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Citra Santoso,CP0001839
5,5110000.0,CUS0009012,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Bima Kurniawan,CP0001201
6,9494000.0,CUS0007860,Medium,0,CUS0003329,ACC00007330,Farah Wibowo,CUS0003329
7,27469000.0,CUS0005278,Low,0,CUS0005801,ACC00012038,Anisa Iskandar,CUS0005801
8,16129000.0,CUS0008434,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Eka Hartono,CP0003878
9,22418000.0,CUS0008497,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,PT Darmawan Synthetic,CP0000835


In [12]:
joined_tables.to_csv(
    PROJECT_ROOT / "notebooks" / "joined_tables.csv",
    index=False,
    encoding="utf-8",
)

## 6. Standarisasi Base Table untuk feature engineering



### 6.1 Create a clean, transaction-grain dataframe


In [ ]:

BASE_REQUIRED_COLUMNS = [
    "transaction_id",
    "transaction_timestamp",
    "transaction_status",
    "amount_idr_equivalent",
    "sender_customer_id",
    "sender_account_id",
    "receiver_party_id",
    "receiver_account_id",
]

missing_columns = sorted(set(BASE_REQUIRED_COLUMNS) - set(joined_tables.columns))
if missing_columns:
    raise ValueError(f"Base table is missing required columns: {missing_columns}")


base_transactions = joined_tables.copy()

base_transactions["transaction_timestamp"] = pd.to_datetime(
    base_transactions["transaction_timestamp"],
    errors="raise",
)
for numeric_column in ["amount", "amount_idr_equivalent", "sender_customer_monthly_income"]:
    base_transactions[numeric_column] = pd.to_numeric(
        base_transactions[numeric_column],
        errors="raise",
    )

id_columns = [
    "transaction_id",
    "sender_customer_id",
    "sender_account_id",
    "receiver_party_id",
    "receiver_account_id",
]
for id_column in id_columns:
    base_transactions[id_column] = base_transactions[id_column].astype("string").str.strip()

base_transactions["transaction_status"] = (
    base_transactions["transaction_status"].astype("string").str.strip().str.title()
)
allowed_statuses = {"Success", "Failed", "Reversed"}
unexpected_statuses = sorted(set(base_transactions["transaction_status"].dropna()) - allowed_statuses)
if unexpected_statuses:
    raise ValueError(f"Unexpected transaction_status values: {unexpected_statuses}")

# Ini Paling Penting tambah 5 kolom biar informasi semakin lengkap. Sisanya cuma vaalidasi sama standarisasi aja dari base tabel transaksi. jadi standarisasi ke timestamp dan nominal ke tipe yang bener
base_transactions["is_success"] = base_transactions["transaction_status"].eq("Success")
base_transactions["is_internal_receiver"] = base_transactions["receiver_party_id"].str.startswith("CUS", na=False)
base_transactions["is_external_receiver"] = base_transactions["receiver_party_id"].str.startswith("CP", na=False)
base_transactions["transaction_date"] = base_transactions["transaction_timestamp"].dt.normalize()
base_transactions["transaction_hour"] = base_transactions["transaction_timestamp"].dt.hour.astype("int8")


assert (
    base_transactions["is_internal_receiver"] ^ base_transactions["is_external_receiver"]
).all(), "Each transaction must resolve to exactly one internal or external receiver party."

standardisation_summary = pd.DataFrame(
    [{
        "rows": len(base_transactions),
        "unique_transaction_ids": base_transactions["transaction_id"].nunique(),
        "successful_transactions": int(base_transactions["is_success"].sum()),
        "internal_receiver_transactions": int(base_transactions["is_internal_receiver"].sum()),
        "external_receiver_transactions": int(base_transactions["is_external_receiver"].sum()),
        "first_transaction": base_transactions["transaction_timestamp"].min(),
        "last_transaction": base_transactions["transaction_timestamp"].max(),
    }]
)
display(standardisation_summary)
base_transactions.head(5)

,rows,unique_transaction_ids,successful_transactions,internal_receiver_transactions,external_receiver_transactions,first_transaction,last_transaction
0,250000,250000,243720,69976,180024,2025-11-03 00:00:18,2026-06-30 23:59:53


,transaction_id,transaction_timestamp,transaction_type,channel,transaction_status,debit_credit,amount,currency,amount_idr_equivalent,purpose_code,...,receiver_customer_id,receiver_account_id,counterparty_id,beneficiary_name,beneficiary_address,is_success,is_internal_receiver,is_external_receiver,transaction_date,transaction_hour
0,TXN0000000001,2025-11-03 00:00:18,SWIFT,Internet,Success,Debit,6211901.22,IDR,6211901.22,OTHER,...,CUS0007767,ACC00003390,INTERNAL_ON_US_TRANSFER,Rizky Chandra,Jl. Prakoso No. 101,True,True,False,2025-11-03,0
1,TXN0000000002,2025-11-03 00:02:26,RTGS,Branch,Success,Debit,1347876.32,IDR,1347876.32,INVESTMENT,...,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0004049,Gita Adinata,101 Synthetic Avenue,True,False,True,2025-11-03,0
2,TXN0000000003,2025-11-03 00:03:18,BI-FAST,ATM,Success,Debit,987476.37,IDR,987476.37,BILL,...,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0003095,Raka Santoso,92 Synthetic Avenue,True,False,True,2025-11-03,0
3,TXN0000000004,2025-11-03 00:04:15,Cash,Mobile,Success,Debit,149.95,USD,2429199.44,FAMILY,...,CUS0000366,ACC00008341,INTERNAL_ON_US_TRANSFER,Indra Adinata,Jl. Gunawan No. 92,True,True,False,2025-11-03,0
4,TXN0000000005,2025-11-03 00:06:48,Transfer,API,Success,Debit,927217.72,IDR,927217.72,TRADE,...,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0001839,Citra Santoso,593 Synthetic Avenue,True,False,True,2025-11-03,0


### 6.2 Validasi Standarisasi tabel

In [8]:
standardisation_checks = pd.Series(
    {
        "transaction_id_is_unique": base_transactions["transaction_id"].is_unique,
        "timestamp_is_parseable": base_transactions["transaction_timestamp"].notna().all(),
        "amount_idr_is_positive": base_transactions["amount_idr_equivalent"].gt(0).all(),
        "sender_income_is_positive": base_transactions["sender_customer_monthly_income"].gt(0).all(),
        "sender_account_is_present": base_transactions["sender_account_id"].notna().all(),
        "receiver_party_is_present": base_transactions["receiver_party_id"].notna().all(),
        "receiver_party_is_classified": (
            base_transactions["is_internal_receiver"]
            | base_transactions["is_external_receiver"]
        ).all(),
    },
    name="passed",
).to_frame()
display(standardisation_checks)
assert standardisation_checks["passed"].all(), "Base-table standardisation validation failed."

,passed
transaction_id_is_unique,True
timestamp_is_parseable,True
amount_idr_is_positive,True
sender_income_is_positive,True
sender_account_is_present,True
receiver_party_is_present,True
receiver_party_is_classified,True


## 7. Feature configuration and ABT working copy

Nilai-nilai di bawah ini pengaturan yang saya setting untuk experiment. Semua pengaturan sengaja dikumpulkan dalam satu cell agar nanti mudah diubah tanpa perlu mengubah ulang logika pembuatan fitur.



In [25]:
from collections import Counter, deque

import numpy as np

FEATURE_CONFIG = {
    # Structuring / Smurfing
    "small_transaction_max_idr": 10_000_000,
   
    "activity_windows": {"1h": "1h", "120m": "120min", "24h": "24h", "7d": "7D"},
    "structuring_candidate_window": "24h",
    "structuring_min_transaction_count": 4,
    "structuring_min_total_idr": 30_000_000,
    # Rapid movement of funds
    "rapid_lookback": pd.Timedelta("24h"),
    "rapid_alert_minutes": 30,
    "rapid_ratio_low": 0.75,
    "rapid_ratio_high": 1.10,
    # Sudden transaction spike
    "spike_window": "30D",
    "spike_min_history": 3,
    "spike_ratio_to_prior_max": 3.0,
    # Dormant account reactivation
    "dormant_days": 60,
    "dormant_large_amount_idr": 150_000_000,
    # Multiple senders to one receiver
    "multiple_senders_candidate_window": "24h",
    "multiple_senders_min_distinct": 4,
    "multiple_senders_min_transaction_count": 4,
}

# Keep the standardized base table unchanged and build features in a separate dataframe.
transaction_feature_abt = base_transactions.copy()
successful_transactions = transaction_feature_abt.loc[
    transaction_feature_abt["is_success"]
].copy()

display(pd.Series(FEATURE_CONFIG, name="value").to_frame())
print(f"Jumlah ABT: {len(transaction_feature_abt):,} baris / {transaction_feature_abt['transaction_id'].nunique():,} ID transaksi")

,value
small_transaction_max_idr,10000000
activity_windows,"{'1h': '1h', '120m': '120min', '24h': '24h', '..."
structuring_candidate_window,24h
structuring_min_transaction_count,4
structuring_min_total_idr,30000000
rapid_lookback,1 days 00:00:00
rapid_alert_minutes,30
rapid_ratio_low,0.75
rapid_ratio_high,1.1
spike_window,30D


Jumlah ABT: 250,000 baris / 250,000 ID transaksi


## 8. Structuring / Smurfing features

Pada bagian ini, dibuat features untuk setiap rekening pengirim untuk periode waktu seperti:

1 jam

120 menit

24 jam

7 hari

Untuk setiap periode tersebut, dihitung:

jumlah transaksi

total nilai transaksi dalam Rupiah

Perhitungan yang sama dilakukan juga buat transaksi yang nilainya berada di bawah batas tertentu. Fitur ini penting untuk mendeteksi Structuring,  ketika dana besar sengaja dipecah menjadi banyak transaksi kecil agar terlihat tidak mencurigakan.

Semua perhitungan window hanya menggunakan riwayat transaksi sebelumnya dengan closed="left", lalu ditambahkan transaksi yang sedang diproses saat ini. Dengan cara ini, data dari masa depan tidak ikut digunakan.

Kandidat Structuring menggunakan fitur transaksi di bawah batas tertentu dalam periode 24 jam. Sementara itu, fitur 1 jam, 120 menit, dan 7 hari tetap disimpan agar nantinya dapat digunakan oleh model atau untuk menyesuaikan aturan deteksi.

In [26]:
def build_sender_window_features(events, feature_prefix):
    
    ordered_events = events.sort_values(
        ["sender_account_id", "transaction_timestamp", "transaction_id"],
        kind="mergesort",
    ).copy()
    feature_frame = pd.DataFrame(index=ordered_events.index)

    for window_label, rolling_window in FEATURE_CONFIG["activity_windows"].items():
        prior_window = (
            ordered_events.groupby("sender_account_id", sort=False)
            .rolling(
                rolling_window,
                on="transaction_timestamp",
                closed="left",
            )["amount_idr_equivalent"]
            .agg(["count", "sum"])
        )

        count_column = f"{feature_prefix}_txn_count_{window_label}"
        amount_column = f"{feature_prefix}_amount_sum_{window_label}_idr"
        # Add the transaction currently being scored to strict-before history.
        feature_frame[count_column] = (
            prior_window["count"].fillna(0).to_numpy(dtype="int32") + 1
        )
        feature_frame[amount_column] = (
            prior_window["sum"].fillna(0.0).to_numpy(dtype="float64")
            + ordered_events["amount_idr_equivalent"].to_numpy(dtype="float64")
        )

    return feature_frame


sender_velocity_features = build_sender_window_features(
    successful_transactions,
    feature_prefix="sender_success",
)


small_successful_transactions = successful_transactions.loc[
    successful_transactions["amount_idr_equivalent"].le(
        FEATURE_CONFIG["small_transaction_max_idr"]
    )
].copy()
subthreshold_features = build_sender_window_features(
    small_successful_transactions,
    feature_prefix="sender_subthreshold",
)

for feature_frame in [sender_velocity_features, subthreshold_features]:
    for feature_name in feature_frame.columns:
        default_value = 0.0 if feature_name.endswith("_idr") else 0
        transaction_feature_abt[feature_name] = default_value
        transaction_feature_abt.loc[feature_frame.index, feature_name] = (
            feature_frame[feature_name].to_numpy()
        )

structuring_candidate_window = FEATURE_CONFIG["structuring_candidate_window"]
transaction_feature_abt["is_structuring_candidate_24h"] = (
    transaction_feature_abt["is_success"]
    & transaction_feature_abt["amount_idr_equivalent"].le(
        FEATURE_CONFIG["small_transaction_max_idr"]
    )
    & transaction_feature_abt[
        f"sender_subthreshold_txn_count_{structuring_candidate_window}"
    ].ge(FEATURE_CONFIG["structuring_min_transaction_count"])
    & transaction_feature_abt[
        f"sender_subthreshold_amount_sum_{structuring_candidate_window}_idr"
    ].ge(FEATURE_CONFIG["structuring_min_total_idr"])
).astype("int8")

display(transaction_feature_abt[[
    "sender_success_txn_count_1h",
    "sender_success_txn_count_120m",
    "sender_success_txn_count_24h",
    "sender_success_txn_count_7d",
    "sender_success_amount_sum_1h_idr",
    "sender_success_amount_sum_120m_idr",
    "sender_success_amount_sum_24h_idr",
    "sender_success_amount_sum_7d_idr",
    "sender_subthreshold_txn_count_120m",
    "sender_subthreshold_txn_count_24h",
    "sender_subthreshold_amount_sum_120m_idr",
    "sender_subthreshold_amount_sum_24h_idr",
    "is_structuring_candidate_24h",
]].describe())

,sender_success_txn_count_1h,sender_success_txn_count_120m,sender_success_txn_count_24h,sender_success_txn_count_7d,sender_success_amount_sum_1h_idr,sender_success_amount_sum_120m_idr,sender_success_amount_sum_24h_idr,sender_success_amount_sum_7d_idr,sender_subthreshold_txn_count_120m,sender_subthreshold_txn_count_24h,sender_subthreshold_amount_sum_120m_idr,sender_subthreshold_amount_sum_24h_idr,is_structuring_candidate_24h
count,250000.000000,250000.000000,250000.000000,250000.000000,2.500000e+05,2.500000e+05,2.500000e+05,2.500000e+05,250000.00000,250000.000000,2.500000e+05,2.500000e+05,250000.000000
mean,0.978064,0.981428,1.043340,1.437300,6.381090e+06,6.505033e+06,6.912856e+06,9.506859e+06,0.84548,0.891564,2.567280e+06,2.707434e+06,0.000108
std,0.167902,0.181050,0.313519,0.722572,2.687049e+07,3.066850e+07,3.227749e+07,3.780229e+07,0.37494,0.449327,2.510642e+06,2.671235e+06,0.010392
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00000,0.000000,0.000000e+00,0.000000e+00,0.000000
25%,1.000000,1.000000,1.000000,1.000000,1.210960e+06,1.213603e+06,1.277134e+06,1.783697e+06,1.00000,1.000000,6.226458e+05,6.446817e+05,0.000000
50%,1.000000,1.000000,1.000000,1.000000,2.724156e+06,2.732091e+06,2.922049e+06,4.325185e+06,1.00000,1.000000,1.818353e+06,1.907635e+06,0.000000
75%,1.000000,1.000000,1.000000,2.000000,6.035082e+06,6.056396e+06,6.523936e+06,9.550072e+06,1.00000,1.000000,3.868628e+06,4.095458e+06,0.000000
max,4.000000,4.000000,6.000000,7.000000,1.076908e+09,1.736852e+09,1.736852e+09,1.739096e+09,4.00000,5.000000,3.862489e+07,4.450343e+07,1.000000


## 9. Rapid Movement of Funds features

Tujuan bagian ini adalah mencari pola **dana masuk ke rekening internal lalu dana keluar lagi dengan cepat**. 

### Apa yang dilakukan code di bawah

1. **Membuat `outbound_events`**  
   Semua transaksi berstatus `Success` diperlakukan sebagai transaksi keluar dari `sender_account_id`. Kolom `base_row_index` disimpan agar hasil feature dapat dikembalikan ke baris transaksi yang tepat di `transaction_feature_abt`.

2. **Membuat `inbound_events`**  
   Code memilih transaksi sukses dengan `is_internal_receiver = True`. Pada transaksi seperti ini, `receiver_account_id` berarti rekening internal yang menerima dana. Kolom tersebut diganti nama menjadi `account_id`, sehingga ia dapat dibandingkan dengan `sender_account_id` pada transaksi keluar.

   Ini diperlukan karena kolom raw `debit_credit` seluruhnya bernilai `Debit`. Arah dana masuk diturunkan dari peran rekening sebagai `receiver_account_id`, bukan dari nilai `debit_credit`.

3. **Mencocokkan inbound terakhir dengan `pd.merge_asof`**  
   Untuk setiap transaksi keluar, `merge_asof` mencari transaksi masuk internal terakhir pada rekening yang sama dalam lookback 24 jam. `direction="backward"` berarti hanya boleh mencari ke masa lalu, sedangkan `allow_exact_matches=False` mencegah transaksi pada timestamp yang sama menjadi histori. Kedua parameter ini menjaga feature tetap leakage-safe.

4. **Membuat feature Rapid Movement**
   - `has_prior_internal_inbound_24h`: `1` bila ada inbound internal sebelumnya dalam 24 jam.
   - `minutes_since_last_internal_inbound`: jeda menit dari inbound terakhir ke transaksi keluar saat ini; `-1` berarti tidak ada inbound yang cocok.
   - `last_internal_inbound_amount_idr`: nominal inbound terakhir; `0` berarti tidak ada inbound yang cocok.
   - `outbound_to_last_inbound_ratio`: nominal keluar ÷ nominal inbound terakhir; `0` berarti rasio tidak dapat dihitung karena tidak ada inbound.

5. **Membuat `is_rapid_movement_candidate`**  
   Flag prototype bernilai `1` jika transaksi sukses memiliki inbound internal sebelumnya, terjadi maksimal 30 menit kemudian, dan rasio nominal keluar terhadap inbound berada pada rentang 0,75–1,10. Seluruh nilai ini ada di `FEATURE_CONFIG` dan dapat diubah tanpa mengubah logika code.



In [27]:
outbound_events = successful_transactions[[
    "transaction_id",
    "transaction_timestamp",
    "sender_account_id",
    "amount_idr_equivalent",
]].copy().rename(
    columns={
        "sender_account_id": "account_id",
        "amount_idr_equivalent": "outbound_amount_idr",
    }
)
outbound_events["base_row_index"] = outbound_events.index

inbound_events = successful_transactions.loc[
    successful_transactions["is_internal_receiver"],
    [
        "transaction_id",
        "transaction_timestamp",
        "receiver_account_id",
        "amount_idr_equivalent",
    ],
].copy().rename(
    columns={
        "transaction_id": "inbound_transaction_id",
        "transaction_timestamp": "inbound_timestamp",
        "receiver_account_id": "account_id",
        "amount_idr_equivalent": "last_internal_inbound_amount_idr",
    }
)

rapid_matches = pd.merge_asof(
    outbound_events.sort_values(
        ["transaction_timestamp", "account_id", "transaction_id"],
        kind="mergesort",
    ),
    inbound_events.sort_values(
        ["inbound_timestamp", "account_id", "inbound_transaction_id"],
        kind="mergesort",
    ),
    by="account_id",
    left_on="transaction_timestamp",
    right_on="inbound_timestamp",
    direction="backward",
    allow_exact_matches=False,
    tolerance=FEATURE_CONFIG["rapid_lookback"],
).set_index("base_row_index")

minutes_since_inbound = (
    rapid_matches["transaction_timestamp"] - rapid_matches["inbound_timestamp"]
).dt.total_seconds().div(60)
prior_inbound_amount = rapid_matches["last_internal_inbound_amount_idr"].fillna(0.0)
outbound_to_inbound_ratio = rapid_matches["outbound_amount_idr"].div(
    prior_inbound_amount.where(prior_inbound_amount.gt(0))
).fillna(0.0)

for feature_name, default_value, feature_values in [
    (
        "has_prior_internal_inbound_24h",
        0,
        rapid_matches["inbound_transaction_id"].notna().astype("int8"),
    ),
    ("minutes_since_last_internal_inbound", -1.0, minutes_since_inbound.fillna(-1.0)),
    ("last_internal_inbound_amount_idr", 0.0, prior_inbound_amount),
    ("outbound_to_last_inbound_ratio", 0.0, outbound_to_inbound_ratio),
]:
    transaction_feature_abt[feature_name] = default_value
    transaction_feature_abt.loc[rapid_matches.index, feature_name] = feature_values.to_numpy()

transaction_feature_abt["is_rapid_movement_candidate"] = (
    transaction_feature_abt["is_success"]
    & transaction_feature_abt["has_prior_internal_inbound_24h"].eq(1)
    & transaction_feature_abt["minutes_since_last_internal_inbound"].between(
        0, FEATURE_CONFIG["rapid_alert_minutes"]
    )
    & transaction_feature_abt["outbound_to_last_inbound_ratio"].between(
        FEATURE_CONFIG["rapid_ratio_low"],
        FEATURE_CONFIG["rapid_ratio_high"],
    )
).astype("int8")

transaction_feature_abt[[
    "has_prior_internal_inbound_24h",
    "minutes_since_last_internal_inbound",
    "outbound_to_last_inbound_ratio",
    "is_rapid_movement_candidate",
]].describe()

,has_prior_internal_inbound_24h,minutes_since_last_internal_inbound,outbound_to_last_inbound_ratio,is_rapid_movement_candidate
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,0.018596,11.908453,0.074721,0.000424
std,0.135094,110.232130,1.865833,0.020587
min,0.000000,-1.000000,0.000000,0.000000
25%,0.000000,-1.000000,0.000000,0.000000
50%,0.000000,-1.000000,0.000000,0.000000
75%,0.000000,-1.000000,0.000000,0.000000
max,1.000000,1439.833333,295.579048,1.000000


## 10. Sudden Transaction Spike features

Buat setiap rekening pengirim jumlah transaksi, rata-rata, median, dan nilai transaksi terbesar dari transaksi berhasil selama 30 hari sebelumnya semuanya dihitung.
Transaksi yang sedang diperiksa tidak ikut dihitung dalam statistik tersebut karena menggunakan `closed="left"`. Setelah itu, nilai transaksi saat ini dibandingkan dengan riwayat transaksi sebelumnya.

Misal riwayat transaksinya belum cukup, semua fitur numerik historis akan diisi pake nilai nol. Selain itu, kolom `has_sufficient_history_30d` digunakan sebagai penanda apakah rekening tersebut memiliki riwayat transaksi 30 hari yang cukup atau tidak.

In [28]:
spike_events = successful_transactions.sort_values(
    ["sender_account_id", "transaction_timestamp", "transaction_id"],
    kind="mergesort",
).copy()

spike_history = (
    spike_events.groupby("sender_account_id", sort=False)
    .rolling(
        FEATURE_CONFIG["spike_window"],
        on="transaction_timestamp",
        closed="left",
    )["amount_idr_equivalent"]
    .agg(["count", "mean", "median", "max"])
)

for statistic, feature_name, default_value in [
    ("count", "prior_success_txn_count_30d", 0),
    ("mean", "prior_amount_mean_30d_idr", 0.0),
    ("median", "prior_amount_median_30d_idr", 0.0),
    ("max", "prior_amount_max_30d_idr", 0.0),
]:
    spike_events[feature_name] = spike_history[statistic].fillna(0.0).to_numpy()
    transaction_feature_abt[feature_name] = default_value
    transaction_feature_abt.loc[spike_events.index, feature_name] = (
        spike_events[feature_name].to_numpy()
    )

transaction_feature_abt["prior_success_txn_count_30d"] = (
    transaction_feature_abt["prior_success_txn_count_30d"].astype("int32")
)
transaction_feature_abt["amount_to_prior_median_ratio_30d"] = np.where(
    transaction_feature_abt["prior_amount_median_30d_idr"].gt(0),
    transaction_feature_abt["amount_idr_equivalent"]
    / transaction_feature_abt["prior_amount_median_30d_idr"],
    0.0,
)
transaction_feature_abt["amount_to_prior_max_ratio_30d"] = np.where(
    transaction_feature_abt["prior_amount_max_30d_idr"].gt(0),
    transaction_feature_abt["amount_idr_equivalent"]
    / transaction_feature_abt["prior_amount_max_30d_idr"],
    0.0,
)
transaction_feature_abt["has_sufficient_history_30d"] = (
    transaction_feature_abt["prior_success_txn_count_30d"].ge(
        FEATURE_CONFIG["spike_min_history"]
    )
).astype("int8")
transaction_feature_abt["is_sudden_spike_candidate"] = (
    transaction_feature_abt["is_success"]
    & transaction_feature_abt["has_sufficient_history_30d"].eq(1)
    & transaction_feature_abt["amount_to_prior_max_ratio_30d"].ge(
        FEATURE_CONFIG["spike_ratio_to_prior_max"]
    )
).astype("int8")

transaction_feature_abt[[
    "prior_success_txn_count_30d",
    "amount_to_prior_median_ratio_30d",
    "amount_to_prior_max_ratio_30d",
    "is_sudden_spike_candidate",
]].describe()

,prior_success_txn_count_30d,amount_to_prior_median_ratio_30d,amount_to_prior_max_ratio_30d,is_sudden_spike_candidate
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,1.879060,2.498397,1.817244,0.019152
std,1.456281,13.248997,11.843644,0.137059
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.112502,0.057853,0.000000
50%,2.000000,0.591135,0.335920,0.000000
75%,3.000000,1.892292,1.176427,0.000000
max,12.000000,2244.508672,2244.508672,1.000000


## 11. Dormant Account Reactivation features

Rekening dianggap tidak aktif berdasarkan riwayat transaksi yang ada di dalam data, bukan berdasarkan `opening_date` atau status rekening.

Untuk setiap transaksi berhasil, sistem akan melihat kapan terakhir kali rekening pengirim melakukan transaksi berhasil sebelum transaksi saat ini.

Jika tidak ditemukan transaksi sebelumnya, ya nilainya `-1`. Nilai ini hanya sebagai penanda bahwa belum ada riwayat aktivitas sebelumnya, dan dibaca bersama kolom `has_prior_successful_sender_activity`.

Jadi, nilai `-1` bukan berarti rekening tidak aktif selama minus satu hari.

In [29]:
# Keep one row per account and timestamp so events at the same timestamp never become each other's history.
activity_timestamps = (
    successful_transactions[["sender_account_id", "transaction_timestamp"]]
    .drop_duplicates()
    .sort_values(["sender_account_id", "transaction_timestamp"], kind="mergesort")
    .copy()
)
activity_timestamps["previous_successful_sender_timestamp"] = (
    activity_timestamps.groupby("sender_account_id", sort=False)["transaction_timestamp"]
    .shift(1)
)

dormant_lookup = successful_transactions[[
    "sender_account_id",
    "transaction_timestamp",
]].merge(
    activity_timestamps,
    on=["sender_account_id", "transaction_timestamp"],
    how="left",
    validate="many_to_one",
)
dormant_lookup.index = successful_transactions.index
prior_gap_days = (
    dormant_lookup["transaction_timestamp"]
    - dormant_lookup["previous_successful_sender_timestamp"]
).dt.total_seconds().div(86_400)

transaction_feature_abt["has_prior_successful_sender_activity"] = 0
transaction_feature_abt["days_since_prior_successful_sender_activity"] = -1.0
transaction_feature_abt.loc[
    dormant_lookup.index, "has_prior_successful_sender_activity"
] = dormant_lookup["previous_successful_sender_timestamp"].notna().astype("int8").to_numpy()
transaction_feature_abt.loc[
    dormant_lookup.index, "days_since_prior_successful_sender_activity"
] = prior_gap_days.fillna(-1.0).to_numpy()

transaction_feature_abt["is_dormant_60d"] = (
    transaction_feature_abt["is_success"]
    & transaction_feature_abt["days_since_prior_successful_sender_activity"].ge(
        FEATURE_CONFIG["dormant_days"]
    )
).astype("int8")
transaction_feature_abt["is_dormant_reactivation_candidate"] = (
    transaction_feature_abt["is_dormant_60d"].eq(1)
    & transaction_feature_abt["amount_idr_equivalent"].ge(
        FEATURE_CONFIG["dormant_large_amount_idr"]
    )
).astype("int8")

transaction_feature_abt[[
    "has_prior_successful_sender_activity",
    "days_since_prior_successful_sender_activity",
    "is_dormant_60d",
    "is_dormant_reactivation_candidate",
]].describe()

,has_prior_successful_sender_activity,days_since_prior_successful_sender_activity,is_dormant_60d,is_dormant_reactivation_candidate
count,250000.000000,250000.000000,250000.000000,250000.000000
mean,0.915520,12.419788,0.011208,0.000404
std,0.278107,13.630045,0.105273,0.020096
min,0.000000,-1.000000,0.000000,0.000000
25%,1.000000,2.726250,0.000000,0.000000
50%,1.000000,8.297303,0.000000,0.000000
75%,1.000000,17.775880,0.000000,0.000000
max,1.000000,168.284444,1.000000,1.000000


## 12. Multiple Senders to One Receiver features


receiver_party_id digunakan biar satu metode yang sama dapat digunakan untuk penerima internal (CUS...) maupun pihak eksternal atau counterparty (CP...).

Untuk setiap periode 1 jam, 120 menit, 24 jam, dan 7 hari, bagian ini menghitung:

 transaction count, total received amount, and distinct sender customers.

Perhitungan dilakukan menggunakan sliding window sederhana dengan Counter. Cara ini dipilih karena rolling().apply(nunique) jauh lebih lambat untuk ukuran data ini.

Jika beberapa transaksi memiliki waktu yang sama persis, transaksi-transaksi tersebut tidak dianggap sebagai riwayat satu sama lain.

Setiap fitur cuma menggunakan aktivitas yang terjadi sebelum transaksi saat ini, lalu menambahkan transaksi saat ini ke dalam perhitungan. Biar sesuai untuk pemantauan transaksi secara real-time.

In [ ]:
def build_receiver_window_features(events):
    
    number_of_rows = len(transaction_feature_abt)
    receiver_events = events[[
        "transaction_id",
        "transaction_timestamp",
        "sender_customer_id",
        "receiver_party_id",
        "amount_idr_equivalent",
    ]].copy()
    receiver_events["base_row_position"] = transaction_feature_abt.index.get_indexer(
        receiver_events.index
    )
    receiver_events = receiver_events.sort_values(
        ["receiver_party_id", "transaction_timestamp", "transaction_id"],
        kind="mergesort",
    )

    feature_arrays = {}
    for window_label in FEATURE_CONFIG["activity_windows"]:
        feature_arrays[f"receiver_txn_count_{window_label}"] = np.zeros(
            number_of_rows, dtype=np.int32
        )
        feature_arrays[f"distinct_senders_to_receiver_{window_label}"] = np.zeros(
            number_of_rows, dtype=np.int32
        )
        feature_arrays[f"receiver_amount_sum_{window_label}_idr"] = np.zeros(
            number_of_rows, dtype=np.float64
        )

    for _, receiver_group in receiver_events.groupby("receiver_party_id", sort=False):
        row_positions = receiver_group["base_row_position"].to_numpy()
        timestamps = receiver_group["transaction_timestamp"].tolist()
        sender_ids = receiver_group["sender_customer_id"].tolist()
        amounts = receiver_group["amount_idr_equivalent"].to_numpy(dtype="float64")

        # Each horizon keeps its own queue, sender counter, and amount total.
        states = {
            label: {"events": deque(), "senders": Counter(), "total": 0.0}
            for label in FEATURE_CONFIG["activity_windows"]
        }
        start_position = 0

        while start_position < len(receiver_group):
            current_timestamp = timestamps[start_position]
            end_position = start_position + 1
            while (
                end_position < len(receiver_group)
                and timestamps[end_position] == current_timestamp
            ):
                end_position += 1

            for window_label, rolling_window in FEATURE_CONFIG["activity_windows"].items():
                state = states[window_label]
                cutoff_timestamp = current_timestamp - pd.Timedelta(rolling_window)
                while state["events"] and state["events"][0][0] < cutoff_timestamp:
                    _, old_sender_id, old_amount = state["events"].popleft()
                    state["senders"][old_sender_id] -= 1
                    if state["senders"][old_sender_id] == 0:
                        del state["senders"][old_sender_id]
                    state["total"] -= old_amount

                historical_txn_count = len(state["events"])
                historical_distinct_sender_count = len(state["senders"])
                historical_total_amount = state["total"]

                for position in range(start_position, end_position):
                    row_position = row_positions[position]
                    sender_is_new = int(sender_ids[position] not in state["senders"])
                    feature_arrays[f"receiver_txn_count_{window_label}"][row_position] = (
                        historical_txn_count + 1
                    )
                    feature_arrays[
                        f"distinct_senders_to_receiver_{window_label}"
                    ][row_position] = historical_distinct_sender_count + sender_is_new
                    feature_arrays[f"receiver_amount_sum_{window_label}_idr"][row_position] = (
                        historical_total_amount + amounts[position]
                    )

            
            for position in range(start_position, end_position):
                for window_label in FEATURE_CONFIG["activity_windows"]:
                    state = states[window_label]
                    state["events"].append(
                        (timestamps[position], sender_ids[position], amounts[position])
                    )
                    state["senders"][sender_ids[position]] += 1
                    state["total"] += amounts[position]

            start_position = end_position

    return pd.DataFrame(feature_arrays, index=transaction_feature_abt.index)

receiver_window_features = build_receiver_window_features(successful_transactions)
for feature_name in receiver_window_features.columns:
    transaction_feature_abt[feature_name] = receiver_window_features[feature_name].to_numpy()

multiple_senders_candidate_window = FEATURE_CONFIG["multiple_senders_candidate_window"]
transaction_feature_abt["is_multiple_senders_candidate_24h"] = (
    transaction_feature_abt["is_success"]
    & transaction_feature_abt[
        f"receiver_txn_count_{multiple_senders_candidate_window}"
    ].ge(FEATURE_CONFIG["multiple_senders_min_transaction_count"])
    & transaction_feature_abt[
        f"distinct_senders_to_receiver_{multiple_senders_candidate_window}"
    ].ge(FEATURE_CONFIG["multiple_senders_min_distinct"])
).astype("int8")

display(transaction_feature_abt[[
    "receiver_txn_count_1h",
    "receiver_txn_count_120m",
    "receiver_txn_count_24h",
    "receiver_txn_count_7d",
    "receiver_amount_sum_1h_idr",
    "receiver_amount_sum_120m_idr",
    "receiver_amount_sum_24h_idr",
    "receiver_amount_sum_7d_idr",
    "distinct_senders_to_receiver_120m",
    "distinct_senders_to_receiver_24h",
    "is_multiple_senders_candidate_24h",
]].describe())

,receiver_txn_count_1h,receiver_txn_count_120m,receiver_txn_count_24h,receiver_txn_count_7d,receiver_amount_sum_1h_idr,receiver_amount_sum_120m_idr,receiver_amount_sum_24h_idr,receiver_amount_sum_7d_idr,distinct_senders_to_receiver_120m,distinct_senders_to_receiver_24h,is_multiple_senders_candidate_24h
count,250000.000000,250000.000000,250000.000000,250000.000000,2.500000e+05,2.500000e+05,2.500000e+05,2.500000e+05,250000.00000,250000.000000,250000.000000
mean,0.980524,0.985584,1.089188,1.749796,6.415353e+06,6.581382e+06,7.262379e+06,1.146383e+07,0.98538,1.088956,0.000524
std,0.175388,0.193495,0.386049,0.990680,2.739569e+07,3.266920e+07,3.381741e+07,4.003171e+07,0.19289,0.385647,0.022885
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,1.000000,1.213460e+06,1.218004e+06,1.326479e+06,2.229398e+06,1.00000,1.000000,0.000000
50%,1.000000,1.000000,1.000000,2.000000,2.730637e+06,2.745807e+06,3.068617e+06,5.522258e+06,1.00000,1.000000,0.000000
75%,1.000000,1.000000,1.000000,2.000000,6.051973e+06,6.089943e+06,6.873091e+06,1.205086e+07,1.00000,1.000000,0.000000
max,4.000000,5.000000,6.000000,9.000000,1.076908e+09,1.910234e+09,1.910234e+09,1.910234e+09,5.00000,6.000000,1.000000


## 14. Save feature ABT



In [33]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
feature_abt_path = PROCESSED_DIR / "transaction_feature_abt.csv"

transaction_feature_abt.to_csv(feature_abt_path, index=False, encoding="utf-8")
pd.DataFrame(
    [{
        "file": feature_abt_path.relative_to(PROJECT_ROOT).as_posix(),
        "rows": len(transaction_feature_abt),
        "columns": len(transaction_feature_abt.columns),
        "size_mb": round(feature_abt_path.stat().st_size / 1_000_000, 2),
    }]
)

,file,rows,columns,size_mb
0,data/processed/transaction_feature_abt.csv,250000,98,200.83


In [17]:
df = pd.read_csv(feature_abt_path)
df.head(10)

,transaction_id,transaction_timestamp,transaction_type,channel,transaction_status,debit_credit,amount,currency,amount_idr_equivalent,purpose_code,...,receiver_txn_count_120m,distinct_senders_to_receiver_120m,receiver_amount_sum_120m_idr,receiver_txn_count_24h,distinct_senders_to_receiver_24h,receiver_amount_sum_24h_idr,receiver_txn_count_7d,distinct_senders_to_receiver_7d,receiver_amount_sum_7d_idr,is_multiple_senders_candidate_24h
0,TXN0000000001,2025-11-03 00:00:18,SWIFT,Internet,Success,Debit,6211901.22,IDR,6211901.22,OTHER,...,1,1,6211901.22,1,1,6211901.22,1,1,6211901.22,0
1,TXN0000000002,2025-11-03 00:02:26,RTGS,Branch,Success,Debit,1347876.32,IDR,1347876.32,INVESTMENT,...,1,1,1347876.32,1,1,1347876.32,1,1,1347876.32,0
2,TXN0000000003,2025-11-03 00:03:18,BI-FAST,ATM,Success,Debit,987476.37,IDR,987476.37,BILL,...,1,1,987476.37,1,1,987476.37,1,1,987476.37,0
3,TXN0000000004,2025-11-03 00:04:15,Cash,Mobile,Success,Debit,149.95,USD,2429199.44,FAMILY,...,1,1,2429199.44,1,1,2429199.44,1,1,2429199.44,0
4,TXN0000000005,2025-11-03 00:06:48,Transfer,API,Success,Debit,927217.72,IDR,927217.72,TRADE,...,1,1,927217.72,1,1,927217.72,1,1,927217.72,0
5,TXN0000000006,2025-11-03 00:09:02,Transfer,Branch,Success,Debit,2097778.34,IDR,2097778.34,INVESTMENT,...,1,1,2097778.34,1,1,2097778.34,1,1,2097778.34,0
6,TXN0000000007,2025-11-03 00:11:11,Cash,Mobile,Success,Debit,569967.57,IDR,569967.57,SALARY,...,1,1,569967.57,1,1,569967.57,1,1,569967.57,0
7,TXN0000000008,2025-11-03 00:14:25,BI-FAST,ATM,Success,Debit,4899100.46,IDR,4899100.46,SALARY,...,1,1,4899100.46,1,1,4899100.46,1,1,4899100.46,0
8,TXN0000000009,2025-11-03 00:15:28,Transfer,API,Failed,Debit,1668090.02,IDR,1668090.02,TRADE,...,0,0,0.00,0,0,0.00,0,0,0.00,0
9,TXN0000000010,2025-11-03 00:16:47,Transfer,API,Success,Debit,1984369.09,IDR,1984369.09,INVESTMENT,...,1,1,1984369.09,1,1,1984369.09,1,1,1984369.09,0


# Berikut di atas merupakan hasil ABT yang sudah jadi.

In [18]:
df.columns

Index(['transaction_id', 'transaction_timestamp', 'transaction_type',
       'channel', 'transaction_status', 'debit_credit', 'amount', 'currency',
       'amount_idr_equivalent', 'purpose_code', 'purpose_description',
       'reference_number', 'source_of_fund', 'destination_bank',
       'destination_country', 'device_id', 'ip_address', 'latitude',
       'longitude', 'sender_customer_id', 'sender_account_id',
       'sender_customer_name', 'sender_customer_address',
       'sender_customer_country', 'sender_customer_monthly_income',
       'sender_customer_occupation', 'sender_customer_segment',
       'sender_customer_risk_rating', 'sender_customer_pep_flag',
       'sender_customer_onboarding_date', 'sender_account_type',
       'sender_account_currency', 'sender_branch_code',
       'sender_account_opening_date', 'sender_account_status',
       'sender_account_risk_level', 'receiver_party_id', 'receiver_party_name',
       'receiver_party_address', 'receiver_party_country',
     